# Classify Raisins with Feature Importance + Hyperparameter Tuning
## Solution Notebook

**Short name (GitHub):** `HypeTune_Py`

Worked answers for the practice skeleton. Numbers below use `train_test_split(..., random_state=19)` so they are reproducible. A default `DecisionTreeClassifier()` with no `random_state` can pick a different `min_samples_split` among the depth-5 ties (Codecademy's published run: `max_depth=5, min_samples_split=3`, CV ≈ 0.867, test ≈ 0.813). With `random_state=19` on the tree we get the same depth and CV, `min_samples_split=2`, test ≈ 0.818.

Not a grading tool for a packing line — morphometrics here are a teaching card.


## Inline cheat-sheet (keep this cell visible)

See also **`HypeTune_Py_Cheatsheet.docx`**.

| Item | Code / rule |
|------|-------------|
| Parameter vs hyperparameter | Coefficients / split thresholds are **learned**. `max_depth`, `C`, `penalty`, `k` are **chosen**. |
| Gini importance | `clf.feature_importances_` after a tree/forest fit with `criterion='gini'`. Biased toward high-cardinality numeric features; ignores correlation. |
| Permutation importance | Shuffle one column, measure drop in score. Model-agnostic, needs a held-out set, slower. `sklearn.inspection.permutation_importance`. |
| Scaled \|coef\| | Only comparable after `StandardScaler`. L1 can zero features. |
| Grid search | Exhaustive Cartesian product of *lists*. `GridSearchCV(est, param_grid, cv=5)`. |
| Random search | Sample `n_iter` draws from *distributions*. `RandomizedSearchCV(est, param_distributions, n_iter=8)`. |
| `uniform(loc, scale)` | Draws on `[loc, loc+scale]`. `uniform(0, 100)` → C ∈ [0, 100]. |
| Attributes after `.fit` | `.best_estimator_`, `.best_params_`, `.best_score_` (mean CV), `.cv_results_`, `.score(X_test, y_test)`. |
| Never | Report `.best_score_` as the final generalisation number. That fold was used to *pick* the hyperparams. |
| `liblinear` | Needed if you still pass `penalty='l1'` on `LogisticRegression`. sklearn ≥ 1.8 prefers `l1_ratio` (0 = L2, 1 = L1). |
| Split once | Freeze `random_state`. Do not retune on the test fold. |


## Flowchart of the desired outcome

![HypeTune flow](hypetune_flowchart.png)

Inspect the balanced 900-row card → estimate which morphometrics actually move the class → exhaust a small tree grid → sample a continuous `C` for logistic regression → confirm on the hold-out fold → poke the knobs in the simulation cell.


## 0. Packages

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from scipy.stats import uniform, loguniform

np.random.seed(19)
plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)


## 1. Load and inspect the raisins table

900 rows, 7 morphometrics (`Area`, `MajorAxisLength`, `MinorAxisLength`, `Eccentricity`, `ConvexArea`, `Extent`, `Perimeter`) plus binary `Class`. Balanced 450 / 450. No missing cells.

![class snapshot](hypetune_data.png)

In [ ]:
df = pd.read_csv("data/Raisin_Dataset.csv")
print("shape:", df.shape)
print("columns:", df.columns.tolist())
print(df.head())
print("\nClass balance:\n", df["Class"].value_counts())
print("\nMissing cells:", int(df.isna().sum().sum()))
print(df.describe().T[["mean", "std", "min", "50%", "max"]].round(3))


## 2. Predictor matrix `X` and target `y`

In [ ]:
X = df.drop(columns="Class")
y = df["Class"]
print("n features:", X.shape[1])
print("n samples:", len(y))
print("class-1 count:", int(y.sum()), "  base rate:", float(y.mean()))


## 3. Train / test split

Codecademy uses `train_test_split(X, y, random_state=19)` with the default 75/25. Freeze that seed for the rest of the notebook.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=19)
print("train:", X_train.shape, " test:", X_test.shape)


## 4. Feature importance — Gini on one tree

Gini impurity at a node is $1 - \sum_k p_k^2$. Gini *gain* is the drop after a split. sklearn stores the total gain attributed to each feature in `feature_importances_`, normalised to sum to 1.

**Watch-outs:** high-cardinality numeric features win ties; correlated size features steal credit from each other.

In [ ]:
# Single tree — Gini impurity reduction, aggregated over every split that used the feature
dt = DecisionTreeClassifier(criterion="gini", random_state=19)
dt.fit(X_train, y_train)
print("default DT test acc:", round(dt.score(X_test, y_test), 4))

gini_dt = pd.Series(dt.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nDT Gini importance:\n", gini_dt.round(4))

ax = gini_dt.sort_values().plot(kind="barh", color="#5dade2")
ax.set_title("Decision-tree Gini importance")
ax.set_xlabel("normalized Gini gain")
plt.tight_layout(); plt.show()


## 5. Aggregate Gini (random forest) and permutation importance

A forest averages Gini across trees that each saw a feature subsample. Permutation importance shuffles one column on the *test* fold and records the drop in accuracy — model-agnostic, correlation-aware in a different way (a redundant feature can look useless once its twin is present).

![importance panel](hypetune_importance.png)

In [ ]:
# Aggregate Gini across a forest + model-agnostic permutation drop on the TEST fold
rf = RandomForestClassifier(n_estimators=200, random_state=19)
rf.fit(X_train, y_train)
print("RF test acc:", round(rf.score(X_test, y_test), 4))

gini_rf = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nRF mean Gini:\n", gini_rf.round(4))

perm = permutation_importance(rf, X_test, y_test, n_repeats=20, random_state=19)
perm_s = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
print("\nPermutation Δacc (RF, test):\n", perm_s.round(4))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
gini_rf.sort_values().plot(kind="barh", ax=axes[0], color="#1abc9c")
axes[0].set_title("RF mean Gini")
perm_s.sort_values().plot(kind="barh", ax=axes[1], color="#e67e22")
axes[1].set_title("Permutation importance (test)")
plt.tight_layout(); plt.show()


## 6. Grid search — Decision Tree

### 6a. Estimator

Lesson constructor: `DecisionTreeClassifier()` with defaults. Depth and split minimum stay as hyperparameters we are about to search.

In [ ]:
tree = DecisionTreeClassifier()   # match the lesson constructor; add random_state=19 for a locked run
print(tree.get_params())


### 6b. `param_grid`

Two lists → 3 × 3 = 9 combinations. Grid search tries every pair.

In [ ]:
parameters = {"min_samples_split": [2, 3, 4], "max_depth": [3, 5, 7]}
print(parameters)
print("grid size:", 3 * 3, "combos  ×  default 5 folds =", 3 * 3 * 5, "fits")


### 6c. `GridSearchCV` + fit

Default `cv=5` so every training row is in a validation fold once. That is 9 × 5 = 45 tree fits, then one refit of the winner on the full training fold (`refit=True`).

In [ ]:
grid = GridSearchCV(tree, parameters)   # cv=5 default
grid.fit(X_train, y_train)
grid


### 6d. Read the winner

`.best_estimator_` is the refitted model. `.best_score_` is the mean CV accuracy of that setting — **not** the number you quote as final performance.

In [ ]:
print("best estimator:", grid.best_estimator_)
print("best params   :", grid.best_params_)
print("best CV score :", round(grid.best_score_, 4))
print("hold-out test :", round(grid.score(X_test, y_test), 4))


### 6e. The 9-cell table

![grid heatmap](hypetune_grid.png)

Depth 5 beats 3 (underfit) and 7 (starts to overfit). `min_samples_split` barely moves the needle on this card.

In [ ]:
grid_table = pd.concat(
    [
        pd.DataFrame(grid.cv_results_["params"]),
        pd.DataFrame(grid.cv_results_["mean_test_score"], columns=["Score"]),
    ],
    axis=1,
)
print(grid_table.sort_values("Score", ascending=False))


## 7. Random search — Logistic Regression

### 7a. Estimator

`solver='liblinear'` is the lesson choice because it supports both L1 and L2. `max_iter=1000` keeps the lbfgs-style convergence warning away.

In [ ]:
lr = LogisticRegression(solver="liblinear", max_iter=1000)
print(lr.get_params())


### 7b. Distributions

A Python list is treated as a discrete uniform. `uniform(loc=0, scale=100)` is continuous on [0, 100]. Draw a few values with `.rvs()` to confirm the range before you search.

In [ ]:
distributions = {"penalty": ["l1", "l2"], "C": uniform(loc=0, scale=100)}
print(distributions)
print("ten draws from C:", np.round(distributions["C"].rvs(10), 2))


### 7c. `RandomizedSearchCV` + fit

`n_iter=8` means eight random `(penalty, C)` pairs, each with 5-fold CV → 40 logistic fits. Pass `random_state` so the draws do not jump between reruns.

In [ ]:
clf = RandomizedSearchCV(lr, distributions, n_iter=8, random_state=19)
clf.fit(X_train, y_train)
clf


### 7d. Winner + table

![random search](hypetune_random.png)

On unscaled raisins the likelihood surface is flat across a wide band of C. Expect several pairs to land within ~0.3 pp of each other.

In [ ]:
print("best estimator:", clf.best_estimator_)
print("best params   :", clf.best_params_)
print("best CV score :", round(clf.best_score_, 4))
print("hold-out test :", round(clf.score(X_test, y_test), 4))

rand_table = pd.concat(
    [
        pd.DataFrame(clf.cv_results_["params"]),
        pd.DataFrame(clf.cv_results_["mean_test_score"], columns=["Accuracy"]),
    ],
    axis=1,
)
print("\n", rand_table.sort_values("Accuracy", ascending=False))


## 8. Alternate code (same question, different route)

Four replacements you can drop in on a new project: locked seed on the tree, log-uniform C, a scaled pipeline so coefficients are feature importances, and a raw double loop so GridSearchCV is not magic.

In [ ]:
print("=" * 60)
print("ALTERNATE A — reproducible tree grid (random_state on the estimator)")
print("=" * 60)
tree_locked = DecisionTreeClassifier(random_state=19)
grid_locked = GridSearchCV(tree_locked, parameters, cv=5)
grid_locked.fit(X_train, y_train)
print(grid_locked.best_params_, "CV", round(grid_locked.best_score_, 4),
      "test", round(grid_locked.score(X_test, y_test), 4))

print("\n" + "=" * 60)
print("ALTERNATE B — log-uniform C + more draws (closer to how C should be searched)")
print("=" * 60)
dist_log = {"penalty": ["l1", "l2"], "C": loguniform(1e-2, 1e2)}
clf_log = RandomizedSearchCV(
    LogisticRegression(solver="liblinear", max_iter=1000),
    dist_log, n_iter=16, random_state=19, cv=5,
)
clf_log.fit(X_train, y_train)
print(clf_log.best_params_, "CV", round(clf_log.best_score_, 4),
      "test", round(clf_log.score(X_test, y_test), 4))

print("\n" + "=" * 60)
print("ALTERNATE C — scaled logistic grid (coefficients become comparable)")
print("=" * 60)
pipe = Pipeline([
    ("sc", StandardScaler()),
    ("lr", LogisticRegression(solver="liblinear", max_iter=2000)),
])
grid_pipe = GridSearchCV(
    pipe,
    {"lr__penalty": ["l1", "l2"], "lr__C": [0.01, 0.1, 1, 10, 100]},
    cv=5,
)
grid_pipe.fit(X_train, y_train)
print(grid_pipe.best_params_, "CV", round(grid_pipe.best_score_, 4),
      "test", round(grid_pipe.score(X_test, y_test), 4))
coef = pd.Series(
    grid_pipe.best_estimator_.named_steps["lr"].coef_[0], index=X.columns
)
print("scaled coefficients (L1 zeros the redundant size features):")
print(coef.sort_values(key=np.abs, ascending=False).round(3))
print("hold-out AUC:", round(roc_auc_score(y_test, grid_pipe.predict_proba(X_test)[:, 1]), 4))

print("\n" + "=" * 60)
print("ALTERNATE D — tiny manual grid (no GridSearchCV) so the mechanics stay visible")
print("=" * 60)
rows = []
for depth in [3, 5, 7]:
    for mss in [2, 3, 4]:
        m = DecisionTreeClassifier(max_depth=depth, min_samples_split=mss, random_state=19)
        m.fit(X_train, y_train)
        rows.append({"max_depth": depth, "min_samples_split": mss,
                     "train": m.score(X_train, y_train), "test": m.score(X_test, y_test)})
print(pd.DataFrame(rows).sort_values("test", ascending=False).round(4))


## 9. More practice

Three short drills that reuse the same split: a third tree hyperparameter, a collinearity drop, and the breast-cancer card from the lesson notebooks.

In [ ]:
print("=" * 60)
print("PRACTICE 1 — add min_samples_leaf to the tree grid")
print("=" * 60)
param2 = {"max_depth": [3, 5, 7], "min_samples_split": [2, 4], "min_samples_leaf": [1, 5, 10]}
g2 = GridSearchCV(DecisionTreeClassifier(random_state=19), param2, cv=5)
g2.fit(X_train, y_train)
print(g2.best_params_, "CV", round(g2.best_score_, 4), "test", round(g2.score(X_test, y_test), 4))

print("\n" + "=" * 60)
print("PRACTICE 2 — drop the two size-collinear columns and re-grid the tree")
print("=" * 60)
drop_cols = ["Area", "ConvexArea"]  # both track Perimeter
Xtr2, Xte2 = X_train.drop(columns=drop_cols), X_test.drop(columns=drop_cols)
g3 = GridSearchCV(DecisionTreeClassifier(random_state=19), parameters, cv=5)
g3.fit(Xtr2, y_train)
print("without Area/ConvexArea:", g3.best_params_,
      "CV", round(g3.best_score_, 4), "test", round(g3.score(Xte2, y_test), 4))

print("\n" + "=" * 60)
print("PRACTICE 3 — sklearn breast-cancer card (the lesson's other table)")
print("=" * 60)
from sklearn.datasets import load_breast_cancer
bc = load_breast_cancer(as_frame=True)
Xb, yb = bc.data, bc.target
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, random_state=19)
lr_bc = LogisticRegression(solver="liblinear", max_iter=2000)
gs_bc = GridSearchCV(lr_bc, {"penalty": ["l1", "l2"], "C": [1, 10, 100]}, cv=5)
gs_bc.fit(Xb_tr, yb_tr)
print("breast-cancer grid:", gs_bc.best_params_,
      "CV", round(gs_bc.best_score_, 4), "test", round(gs_bc.score(Xb_te, yb_te), 4))


## 10. Simulation — change a few knobs

![simulation](hypetune_simulation.png)

Edit `N_ITER`, `TRAIN_N`, `FLIP_P`, `N_JUNK` and re-run. The three sweep charts stay as a map of the neighbourhood around your current setting.

In [ ]:
# --- knobs (edit these and re-run) ---
N_ITER = 8          # RandomizedSearchCV draws
TRAIN_N = 675       # cap the training fold (max 675)
FLIP_P = 0.00       # fraction of training labels to flip
N_JUNK = 0          # extra N(0,1) noise columns
RS = 19

rng = np.random.RandomState(RS)
Xtr = X_train.iloc[:TRAIN_N].copy()
ytr = y_train.iloc[:TRAIN_N].copy()
Xte = X_test.copy()

if N_JUNK > 0:
    for j in range(N_JUNK):
        Xtr[f"junk_{j}"] = rng.normal(size=len(Xtr))
        Xte[f"junk_{j}"] = rng.normal(size=len(Xte))

if FLIP_P > 0:
    y_arr = ytr.to_numpy().copy()
    k = int(FLIP_P * len(y_arr))
    pick = rng.choice(np.arange(len(y_arr)), size=k, replace=False)
    y_arr[pick] = 1 - y_arr[pick]
    ytr = pd.Series(y_arr, index=ytr.index)

grid_s = GridSearchCV(
    DecisionTreeClassifier(random_state=RS),
    {"max_depth": [3, 5, 7], "min_samples_split": [2, 3, 4]},
    cv=5,
)
grid_s.fit(Xtr, ytr)

rand_s = RandomizedSearchCV(
    LogisticRegression(solver="liblinear", max_iter=2000),
    {"penalty": ["l1", "l2"], "C": uniform(0, 100)},
    n_iter=N_ITER, random_state=RS, cv=5,
)
rand_s.fit(Xtr, ytr)

print(f"knobs  N_ITER={N_ITER}  TRAIN_N={TRAIN_N}  FLIP_P={FLIP_P}  N_JUNK={N_JUNK}")
print("DT grid   best", grid_s.best_params_,
      "CV", round(grid_s.best_score_, 4), "test", round(grid_s.score(Xte, y_test), 4))
print("LR random best", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in rand_s.best_params_.items()},
      "CV", round(rand_s.best_score_, 4), "test", round(rand_s.score(Xte, y_test), 4))

# small sweep so the chart updates with the current knobs as the baseline
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6))

niters = [4, 8, 16, 32]
cv_curve, te_curve = [], []
for n in niters:
    rs = RandomizedSearchCV(
        LogisticRegression(solver="liblinear", max_iter=2000),
        {"penalty": ["l1", "l2"], "C": uniform(0, 100)},
        n_iter=n, random_state=RS, cv=5,
    )
    rs.fit(X_train, y_train)
    cv_curve.append(rs.best_score_); te_curve.append(rs.score(X_test, y_test))
axes[0].plot(niters, cv_curve, "o-", label="best CV")
axes[0].plot(niters, te_curve, "s--", label="test")
axes[0].axvline(N_ITER, color="red", ls=":", label="current N_ITER")
axes[0].set_title("n_iter sweep"); axes[0].set_xlabel("n_iter"); axes[0].legend(fontsize=8)

ns = [150, 300, 450, 675]
n_acc = []
for n in ns:
    g = GridSearchCV(DecisionTreeClassifier(random_state=RS),
                     {"max_depth": [3, 5, 7], "min_samples_split": [2, 3, 4]}, cv=5)
    g.fit(X_train.iloc[:n], y_train.iloc[:n])
    n_acc.append(g.score(X_test, y_test))
axes[1].plot(ns, n_acc, "o-", color="#e67e22")
axes[1].axvline(TRAIN_N, color="red", ls=":")
axes[1].set_title("train n sweep"); axes[1].set_xlabel("train n")

flips = [0.0, 0.05, 0.10, 0.20]
f_acc = []
y_base = y_train.to_numpy()
for p in flips:
    yn = y_base.copy()
    k = int(p * len(yn))
    if k:
        pick = rng.choice(np.arange(len(yn)), size=k, replace=False)
        yn[pick] = 1 - yn[pick]
    g = GridSearchCV(DecisionTreeClassifier(random_state=RS),
                     {"max_depth": [3, 5, 7], "min_samples_split": [2, 3, 4]}, cv=5)
    g.fit(X_train, yn)
    f_acc.append(g.score(X_test, y_test))
axes[2].plot([p * 100 for p in flips], f_acc, "o-", color="#c0392b")
axes[2].axvline(FLIP_P * 100, color="red", ls=":")
axes[2].set_title("label-flip sweep"); axes[2].set_xlabel("flip %")
plt.tight_layout(); plt.show()


## 11. Audience rewrite

Use the two attached PDFs (*What to Consider When Considering the Audience*, *Audience and Situation Analysis*). Four cuts of the same result: Experts / Technicians / Executives / Nonspecialists.

In [ ]:
print("""EXPERTS (ML / stats)
Gini gain on a single tree over-credits Perimeter (0.59) because Area, ConvexArea and
Perimeter are near-collinear size measures. Forest-averaged Gini spreads that mass
(Perimeter 0.25, MajorAxisLength 0.22, ConvexArea 0.16, Area 0.15). Permutation on the
hold-out fold collapses onto Perimeter (Δacc ≈ 0.15); the rest are indistinguishable
from noise. The 3×3 tree grid is too coarse to separate min_samples_split at depth 5 —
three cells tie at CV 0.867. Logistic random search on unscaled features is essentially
flat across C ∈ (1, 85): test acc stays at 0.88. A scaled L1 pipeline zeros the redundant
size block and keeps Perimeter (coef ≈ +5.5). Report test, not CV.

TECHNICIANS (QC / packing-line)
Freeze the split (rs=19). Fit the published tree grid {max_depth: 3/5/7,
min_samples_split: 2/3/4}. Take best_estimator_ and score the 225-row hold-out
(~0.81–0.82). Do not retune after looking at that number. Perimeter is the measurement
to keep clean on the camera; Area and ConvexArea are backup size readings.

EXECUTIVES
Two varieties, 900 images, 50/50. A 5-deep tree and a lightly regularised logistic
both land in the mid-80s on new raisins. Extra search budget (n_iter 4 → 32) does not
buy accuracy on this card. The decision that moves the number is 'measure perimeter
well', not 'search a bigger grid'.

NONSPECIALISTS
We looked at photos of two kinds of raisins and asked a simple model which
measurements tell them apart. The outline length (perimeter) does most of the work.
We tried a handful of 'how bushy can the rule-tree get' settings and picked the one
that held up when we hid 25% of the photos. It was right about four times out of five.
That is a teaching result, not a factory spec.
""")


## 12. Takeaways

In [ ]:
print("""1. Hyperparameters are chosen, parameters are learned. max_depth and C do not come from .fit.
2. GridSearchCV exhausts a list; RandomizedSearchCV samples a distribution n_iter times. Same attributes after .fit.
3. .best_score_ is a CV mean used to pick the setting. Generalisation is .score(X_test, y_test).
4. Gini on one tree is cheap and biased toward numeric, high-cardinality, correlated features. Check a forest and a permutation drop.
5. On this card Perimeter dominates; Area / ConvexArea are size echoes. L1 on scaled inputs zeros them.
6. A 3×3 grid already saturates. Bigger n_iter did not lift hold-out accuracy. Label noise and tiny n do.
7. Match the solver to the penalty (liblinear for L1) or switch to the l1_ratio API on sklearn ≥ 1.8.
""")
